# Generating the comparision of different architectures of WGANs across steps

#### clone repo (lsun branch which i worked on)

In [ ]:
!git clone -b lsun https://github.com/uday-kalyan-s/WassersteinGAN.git
%cd WassersteinGAN

#### imports

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from scipy.signal import medfilt
import os

### define functions to plot

In [ ]:
SMOOTH_KERNEL = 51
FIG_SIZE = (14, 5)
IMAGE_WIDTH_RATIO = 0.018
IMAGE_HEIGHT = 0.12

def plot_experiment(folder, title="", caption=None, max_iter=24000):
    # load data
    with open(f"{folder}/loss_log.json") as f:
        data = json.load(f)
    iters = np.array(data["iterations"])
    loss = np.array(data["loss_D"])

    # clip to maximum iteration in case trained anything beyond 24k
    mask = iters <= max_iter
    iters = iters[mask]
    loss = loss[mask]

    # medium filter mentioned in paper
    loss = medfilt(-loss, SMOOTH_KERNEL)

    y_min = loss.min()
    y_max = loss.max()
    y_range = y_max - y_min

    # Row of images sits above the curve
    IMG_TOP    = y_max + y_range * 0.55   # top edge of image row
    IMG_BOTTOM = y_max + y_range * 0.15   # bottom edge of image row (for arrow tip reference)
    img_height = IMG_TOP - IMG_BOTTOM

    # plot the curve
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    ax.plot(iters, loss, color='blue', linewidth=1.5)

    # define intervals as taken every 2k steps
    points = list(range(2000, max_iter + 1, 2000))
    x_total = iters[-1] - iters[0]
    img_half_width = x_total * IMAGE_WIDTH_RATIO * 3

    for p in points:
        img_path = f"{folder}/fake_samples_single_{p}.png"
        if not os.path.exists(img_path):
            continue

        # y value on the curve at this iteration
        idx = np.argmin(np.abs(iters - p))
        y_curve = loss[idx]

        # draw arrow pointing to image from graph pt
        ax.annotate(
            "",
            xy=(p, y_curve),            # arrowhead lands on the curve
            xytext=(p, IMG_BOTTOM),     # tail starts at bottom of image row
            arrowprops=dict(
                arrowstyle="->,head_width=0.3,head_length=0.015",
                color="black",
                lw=0.8,
                shrinkA=0,              # no gap at tail
                shrinkB=2,              # tiny gap before touching the curve
            ),
        )

        # put image
        img = mpimg.imread(img_path)
        ax.imshow(
            img,
            extent=( # defined as rectangle points for drawing image
                p - img_half_width,
                p + img_half_width,
                IMG_BOTTOM,
                IMG_TOP,
            ),
            aspect='auto',
            zorder=3,
        )

    ax.set_xlabel("Generator iterations")
    ax.set_ylabel("Wasserstein estimate")
    ax.set_title(title)
    ax.set_xlim(0, max_iter)
    ax.set_ylim(y_min - y_range * 0.1, IMG_TOP + img_height * 0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # insert caption for image
    if caption:
        fig.text(0.5, -0.04, caption, ha="center", fontsize=11)

    plt.tight_layout()
    plt.show()

## WGAN with DCGAN for both generator and discriminator

In [ ]:
!python main.py --dataset lsun --dataroot "/kaggle/input/lsun-bedroom-64x64-10perc" --batchSize 64 --workers 4 --cuda --experiment DCGAN_WGAN

In [ ]:
plot_experiment(
    folder="DCGAN_WGAN",
    title="DCGAN_WGAN",
    caption="using DCGAN"
)

## WGAN with DCGAN for discriminator and MLP (4 layers, 512 hidden units per layer) for Generator

In [ ]:
!python main.py --dataset lsun --dataroot "/kaggle/input/lsun-bedroom-64x64-10perc" --batchSize 64 --workers 4 --cuda --experiment MLPGEN_WGAN --mlp_G --ngf 512

In [ ]:
plot_experiment(
    folder="MLPGEN_WGAN",
    title="MLPGEN_WGAN",
    caption="using MLP for generator"
)

## WGAN with MLP (4 layers, 512 hidden units per layer) for Generator and Discriminator

In [ ]:
!python main.py --dataset lsun --dataroot "/kaggle/input/lsun-bedroom-64x64-10perc" --batchSize 64 --workers 4 --cuda --experiment MLP_WGAN --mlp_G --ngf 512 --mlp_D --ndf 512 --lrD 0.001 --lrG 0.001

In [ ]:
plot_experiment(
    folder="MLP_WGAN",
    title="MLP_WGAN",
    caption="using MLP for both generator and discriminator with high learning rates"
)